# Incremental

**Content, never modification time.** A `git checkout` rewrites mtimes on identical files, and a restored build cache writes *older* ones than were recorded. The first wastes a full rescan; the second silently skips a file that really changed.

> **Every cell in this notebook runs.** They are generated from
> [`tools/notebooks/spec.py`](../tools/notebooks/spec.py) and executed by CI, so a
> cell that cannot run does not reach a commit. Change a cell, re-run it, and the
> page is yours — that is what it is for.


Hashing costs a read of every file, which sounds like it defeats the purpose —
but reading a file is not the expensive part of a scan. Parsing it, resolving
identities and running twenty-nine discoverers over it is. A fingerprint pass
reads bytes and does nothing else with them.

The second idea is the one this notebook is really about: **the engine never
says anything about a file it did not read.** That sounds obvious. It was not
true, and the way it failed is instructive.

In [ ]:
# --- setup: works locally, on Binder, and on Colab -------------------------
import subprocess, sys, pathlib

def _ensure_installed():
    """Install the package if it is not importable. No-op when it already is."""
    try:
        import slpie, gratimos          # noqa: F401
        return pathlib.Path(slpie.__file__).parent.parent
    except ModuleNotFoundError:
        pass
    here = pathlib.Path.cwd()
    root = next(
        (p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), None,
    )
    if root is None:                     # Colab: no checkout, so fetch one
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Reimain/Macropol-s.git", "/content/Macropol-s"],
            check=True,
        )
        root = pathlib.Path("/content/Macropol-s")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)],
                   check=True)
    sys.path.insert(0, str(root))
    return root

ROOT = _ensure_installed()
print("package root:", ROOT)

import slpie
print("slpie", slpie.__version__)

## Fingerprint a tree

In [ ]:
import pathlib, tempfile
from slpie.incremental import Fingerprint, Watcher

WORK = pathlib.Path(tempfile.mkdtemp(prefix="slpie-nb-"))
tree = WORK / "tree"
tree.mkdir()
for index in range(10):
    (tree / f"module_{index}.py").write_text(f"VALUE = {index}\n")

before = Fingerprint.of(tree)
print(before)
print("tree digest:", before.digest[:32])

## Change one file

In [ ]:
(tree / "module_3.py").write_text("VALUE = 999   # changed\n")
(tree / "module_new.py").write_text("VALUE = None\n")
(tree / "module_7.py").unlink()

after = Fingerprint.of(tree)
delta = after.compare(before)

print(delta)
print()
print("added:    ", [u.rsplit('/', 1)[-1] for u in delta.added])
print("changed:  ", [u.rsplit('/', 1)[-1] for u in delta.changed])
print("removed:  ", [u.rsplit('/', 1)[-1] for u in delta.removed])
print("unchanged:", delta.unchanged)
print()
print("a rescan must re-read:", len(delta.to_read), "file(s), not 10")

## The defect this was built to fix

A fingerprint used to silently drop any file it could not read, was told not to read, or ran out of budget for. `compare()` then reported those files as **removed** — because absent from a fingerprint is indistinguishable from absent from disk.

A rescan acting on that delta retires the graph nodes drawn from files that are still there and perfectly fine. Measured on this same ten-file tree: a size limit of zero reported all ten as removed.

In [ ]:
full = Fingerprint.of(tree)

# A budget that reads nothing at all.
partial = Fingerprint.of(tree, max_bytes=0, strict=False)
delta = partial.compare(full)

print("files this pass read:", len(partial))
print()
print("removed: ", len(delta.removed), " <- would have been 9, retiring the graph")
print("unknown: ", len(delta.unknown), " <- neither refreshed nor retired")
print("trustworthy:", delta.trustworthy)
print()
print(delta)

`unknown` is the whole fix. Those files are left exactly as the graph last knew them, because nobody looked at them — which is the only honest answer.

## Two modes

**Strict** is the default and what production runs. It refuses rather than handing back a delta that looks complete and is not.

In [ ]:
from slpie.incremental import IncompleteFingerprint, TruncatedWalk

try:
    Fingerprint.of(tree, max_bytes=0, strict=True)
except IncompleteFingerprint as error:
    print("REFUSED — and the exception carries its fields, not just a message:")
    print()
    print("  root:    ", error.root)
    print("  reasons: ", error.reasons)
    print("  files:   ", len(error.skipped))
    print()
    print(str(error)[:700])

A caller acts on `error.skipped` rather than parsing prose out of `str(e)` — the shape [python-rope](https://github.com/python-rope/rope) uses for `ModuleSyntaxError(filename, lineno, message)`.

A walk that hits its *file limit* raises a different exception, because it scales differently: it cannot name what it missed without recording one object per unreached file, and on a two-million-file monorepo that is the memory failure the spill tier exists to prevent.

In [ ]:
try:
    Fingerprint.of(tree, limit=3, strict=True)
except TruncatedWalk as error:
    print("limit:", error.limit)
    print()
    print(str(error)[:520])

**Lenient** is for development. It records every skip with the detail needed to fix it, and reports the affected files as unknown.

In [ ]:
lenient = Fingerprint.of(tree, max_bytes=0, strict=False)
print(lenient.explain_skips())

## Planning a rescan before paying for one

In [ ]:
baseline = WORK / "baseline.json"
watcher = Watcher(tree, baseline=baseline)
watcher.commit()                      # record the current state

(tree / "module_1.py").write_text("VALUE = 42   # touched\n")

plan = watcher.plan()
print(plan.render())
print("worth rescanning incrementally:", plan.worth_it)
print("proportion of the tree that moved:", f"{plan.proportion:.0%}")

Past about half the tree, a full rescan is the cheaper answer — the bookkeeping to retire and re-derive most of a graph costs more than building it once. `Plan.proportion` reports that up front rather than pretending otherwise.

## Through the verb

In [ ]:
from slpie.compose import Composition, Context, registry

verbs = registry()
result = Composition.read(f"changed --path {tree} --lenient", verbs=verbs).run(
    Context(root=str(tree)),
)
print(result.flow.facts["changed"])
print("trustworthy:", result.flow.facts["trustworthy"])

In [ ]:
# Scratch cell — make an unreadable file and watch strict mode refuse.
import os
locked = tree / "locked.py"
locked.write_text("SECRET = 1\n")
os.chmod(locked, 0o000)

try:
    Fingerprint.of(tree, strict=True)
    print("(running as root — permissions do not apply, so nothing was skipped)")
except IncompleteFingerprint as error:
    print("refused:", error.reasons, "|", len(error.skipped), "file(s)")
finally:
    os.chmod(locked, 0o644)